# Phase 2: Split inference with separate-process middle stage

Same correctness experiment as Phase 1, but the transformer blocks run in a **separate Python process** (middle worker on CPU). Only hidden states cross the boundary via stdin/stdout with pickle; token IDs and logits stay local.

In [1]:
import os
import sys
import pickle
import subprocess
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
DTYPE = torch.float32
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
PROMPT = "Hey! How are you feeling today?"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=DTYPE).to(DEVICE)
model.eval()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [2]:
base = getattr(model, "model", model)
embed_module = getattr(base, "embed_tokens", None) or getattr(base, "embed", None)
layers_list = getattr(base, "layers", None)
final_norm = getattr(base, "norm", None)
lm_head_module = getattr(model, "lm_head", None)

assert embed_module is not None and layers_list is not None
assert final_norm is not None and lm_head_module is not None

num_blocks = len(layers_list)
print(f"embed_tokens, {num_blocks} blocks, final norm, lm_head")

embed_tokens, 24 blocks, final norm, lm_head


In [3]:
inputs = tokenizer(PROMPT, return_tensors="pt").to(DEVICE)
input_ids_baseline = inputs["input_ids"].clone()
num_new_tokens = 16

with torch.no_grad():
    for _ in range(num_new_tokens):
        outputs = model(input_ids=input_ids_baseline, use_cache=False)
        next_token_id = outputs.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        input_ids_baseline = torch.cat([input_ids_baseline, next_token_id], dim=-1)

generated_ids_baseline = input_ids_baseline[0, inputs["input_ids"].shape[1]:].tolist()
decoded_baseline = tokenizer.decode(generated_ids_baseline, skip_special_tokens=True)
print("Baseline generated token IDs:", generated_ids_baseline)
print("Baseline decoded:", decoded_baseline)

Baseline generated token IDs: [8713, 498, 8266, 1661, 30, 358, 2776, 537, 2704, 1128, 498, 3076, 553, 330, 18536, 1]
Baseline decoded:  Are you feeling good? I'm not sure what you mean by "good"


## Phase 2: Middle stage in a separate process

The notebook keeps **embedding**, **final norm**, and **lm_head** local (trusted). The **transformer blocks** run in a subprocess (`middle_worker.py`) on CPU. Each decode step: (1) compute embeddings locally, (2) send hidden states to the worker via pickle on stdin, (3) worker runs the layers and writes hidden states to stdout, (4) notebook reads the result, (5) apply norm + lm_head locally and take greedy next token. Only hidden-state tensors cross the boundary; token IDs and logits never leave the notebook process.

In [4]:
from multiprocessing.shared_memory import SharedMemory

HIDDEN_SIZE = 896
MAX_SEQ = 2048
max_nbytes = 1 * MAX_SEQ * HIDDEN_SIZE * 4
shm_in = SharedMemory(create=True, size=max_nbytes)
shm_out = SharedMemory(create=True, size=max_nbytes)
p2c_r, p2c_w = os.pipe()
c2p_r, c2p_w = os.pipe()

worker_script = os.path.join(os.getcwd(), "middle_worker.py")
proc = subprocess.Popen(
    [sys.executable, worker_script, MODEL_NAME, str(p2c_r), str(c2p_w), shm_in.name, shm_out.name, str(max_nbytes)],
    pass_fds=[p2c_r, c2p_w],
)
os.close(p2c_r)
os.close(c2p_w)
pipe_to_child = os.fdopen(p2c_w, "wb")
pipe_from_child = os.fdopen(c2p_r, "rb")

def send_hidden_receive_hidden(hidden_states):
    t = hidden_states.cpu().float()
    nbytes = t.numel() * t.element_size()
    memoryview(shm_in.buf)[:nbytes][:] = t.numpy().tobytes()
    pickle.dump((tuple(t.shape), nbytes), pipe_to_child)
    pipe_to_child.flush()
    out_shape, out_nbytes = pickle.load(pipe_from_child)
    out = torch.frombuffer(memoryview(shm_out.buf)[:out_nbytes], dtype=torch.float32).reshape(out_shape).clone()
    return out.to(DEVICE)

input_ids_split = inputs["input_ids"].clone()
with torch.no_grad():
    for _ in range(num_new_tokens):
        hidden_after_embed = embed_module(input_ids_split)
        hidden_after_middle = send_hidden_receive_hidden(hidden_after_embed)
        hidden_after_norm = final_norm(hidden_after_middle)
        logits_local = lm_head_module(hidden_after_norm[:, -1:, :])
        next_token_id = logits_local.argmax(dim=-1)
        input_ids_split = torch.cat([input_ids_split, next_token_id], dim=-1)

pickle.dump(None, pipe_to_child)
pipe_to_child.flush()
proc.wait()
pipe_to_child.close()
pipe_from_child.close()
shm_in.close()
shm_out.close()
shm_in.unlink()
shm_out.unlink()

generated_ids_split = input_ids_split[0, inputs["input_ids"].shape[1]:].tolist()
decoded_split = tokenizer.decode(generated_ids_split, skip_special_tokens=True)
print("Split (Phase 2) generated token IDs:", generated_ids_split)
print("Split decoded:", decoded_split)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 4112.77it/s]


Split (Phase 2) generated token IDs: [8713, 498, 8266, 1661, 30, 358, 2776, 537, 2704, 1128, 498, 3076, 553, 330, 18536, 1]
Split decoded:  Are you feeling good? I'm not sure what you mean by "good"


/opt/homebrew/anaconda3/envs/quantenv/lib/python3.11/multiprocessing/resource_tracker.py:254: UserWarning: resource_tracker: There appear to be 2 leaked shared_memory objects to clean up at shutdown
  warnings.warn('resource_tracker: There appear to be %d '
/opt/homebrew/anaconda3/envs/quantenv/lib/python3.11/multiprocessing/resource_tracker.py:267: UserWarning: resource_tracker: '/psm_00c4627e': [Errno 2] No such file or directory: '/psm_00c4627e'
  warnings.warn('resource_tracker: %r: %s' % (name, e))
/opt/homebrew/anaconda3/envs/quantenv/lib/python3.11/multiprocessing/resource_tracker.py:267: UserWarning: resource_tracker: '/psm_d70ed6e8': [Errno 2] No such file or directory: '/psm_d70ed6e8'
  warnings.warn('resource_tracker: %r: %s' % (name, e))


In [5]:
match = generated_ids_baseline == generated_ids_split
print("Token-by-token comparison:")
for i in range(num_new_tokens):
    status = "ok" if generated_ids_baseline[i] == generated_ids_split[i] else "MISMATCH"
    print(f"  {i}: baseline={generated_ids_baseline[i]}, split={generated_ids_split[i]} [{status}]")
print("\nPASS" if match else "FAIL")
print("\nBaseline decoded:", decoded_baseline)
print("Split decoded:    ", decoded_split)

Token-by-token comparison:
  0: baseline=8713, split=8713 [ok]
  1: baseline=498, split=498 [ok]
  2: baseline=8266, split=8266 [ok]
  3: baseline=1661, split=1661 [ok]
  4: baseline=30, split=30 [ok]
  5: baseline=358, split=358 [ok]
  6: baseline=2776, split=2776 [ok]
  7: baseline=537, split=537 [ok]
  8: baseline=2704, split=2704 [ok]
  9: baseline=1128, split=1128 [ok]
  10: baseline=498, split=498 [ok]
  11: baseline=3076, split=3076 [ok]
  12: baseline=553, split=553 [ok]
  13: baseline=330, split=330 [ok]
  14: baseline=18536, split=18536 [ok]
  15: baseline=1, split=1 [ok]

PASS

Baseline decoded:  Are you feeling good? I'm not sure what you mean by "good"
Split decoded:      Are you feeling good? I'm not sure what you mean by "good"
